In [1]:
from __future__ import annotations

import os
from getpass import getpass

import weaviate
import weaviate.classes as wvc

from weaviate.classes.aggregate import GroupByAggregate
from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery, Metrics

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
COLLECTION_NAME = "CourseEvent"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

events = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="description",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="topic",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="difficulty",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="source",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="duration_minutes",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="quality_score",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="completed",
            data_type=wvc.config.DataType.BOOL,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: CourseEvent


In [5]:
course_events = [
    {
        "title": "Docker Weaviate Setup",
        "description": "Started a local Weaviate instance in Docker and connected from Python notebooks.",
        "topic": "docker",
        "difficulty": "beginner",
        "source": "local",
        "duration_minutes": 45,
        "quality_score": 8.0,
        "completed": True,
    },
    {
        "title": "Cloud Connection",
        "description": "Connected Python client to Weaviate Cloud using cluster URL, Weaviate API key and OpenAI API key.",
        "topic": "cloud",
        "difficulty": "beginner",
        "source": "cloud",
        "duration_minutes": 35,
        "quality_score": 8.5,
        "completed": True,
    },
    {
        "title": "CRUD Operations",
        "description": "Created, read, updated and deleted objects in Weaviate collections using UUIDs.",
        "topic": "crud",
        "difficulty": "beginner",
        "source": "cloud",
        "duration_minutes": 50,
        "quality_score": 8.7,
        "completed": True,
    },
    {
        "title": "Vector Search",
        "description": "Used near_text queries to find semantically similar objects based on embeddings.",
        "topic": "vector_search",
        "difficulty": "intermediate",
        "source": "cloud",
        "duration_minutes": 60,
        "quality_score": 9.0,
        "completed": True,
    },
    {
        "title": "BM25 Keyword Search",
        "description": "Used BM25 to search by exact keywords, technical names, categories and specific terms.",
        "topic": "bm25",
        "difficulty": "intermediate",
        "source": "cloud",
        "duration_minutes": 40,
        "quality_score": 8.2,
        "completed": True,
    },
    {
        "title": "Hybrid Search",
        "description": "Combined vector search and BM25 keyword search using alpha to balance semantic and lexical ranking.",
        "topic": "hybrid_search",
        "difficulty": "intermediate",
        "source": "cloud",
        "duration_minutes": 65,
        "quality_score": 9.1,
        "completed": True,
    },
    {
        "title": "Generative Search",
        "description": "Used grouped_task and single_prompt to generate answers from retrieved Weaviate objects.",
        "topic": "generative_search",
        "difficulty": "intermediate",
        "source": "cloud",
        "duration_minutes": 70,
        "quality_score": 9.3,
        "completed": True,
    },
    {
        "title": "Text-to-Image Search",
        "description": "Used local CLIP embeddings and self-provided vectors to search images in Weaviate Cloud.",
        "topic": "image_search",
        "difficulty": "advanced",
        "source": "cloud",
        "duration_minutes": 90,
        "quality_score": 9.4,
        "completed": True,
    },
    {
        "title": "Focused RAG",
        "description": "Built a focused RAG collection from Weaviate notebooks and Markdown concept notes.",
        "topic": "rag",
        "difficulty": "advanced",
        "source": "cloud",
        "duration_minutes": 110,
        "quality_score": 9.6,
        "completed": True,
    },
    {
        "title": "Named Vectors",
        "description": "Created multiple named vectors for the same object using different source properties.",
        "topic": "named_vectors",
        "difficulty": "advanced",
        "source": "cloud",
        "duration_minutes": 75,
        "quality_score": 9.0,
        "completed": True,
    },
    {
        "title": "Cross-References",
        "description": "Connected review objects to product objects using reference properties between collections.",
        "topic": "cross_references",
        "difficulty": "advanced",
        "source": "cloud",
        "duration_minutes": 80,
        "quality_score": 8.8,
        "completed": True,
    },
    {
        "title": "Multi-Tenancy",
        "description": "Created tenant-isolated data spaces inside one shared collection for SaaS-style applications.",
        "topic": "multi_tenancy",
        "difficulty": "advanced",
        "source": "cloud",
        "duration_minutes": 85,
        "quality_score": 9.2,
        "completed": True,
    },
    {
        "title": "Semantic Recommendations",
        "description": "Combined semantic search with numeric filters, boolean filters and generative explanations.",
        "topic": "recommendations",
        "difficulty": "advanced",
        "source": "cloud",
        "duration_minutes": 95,
        "quality_score": 9.5,
        "completed": True,
    },
    {
        "title": "AI Enrichment Pipeline",
        "description": "Generated summaries, categories, priorities and actions for support tickets, then saved them back to Weaviate.",
        "topic": "ai_enrichment",
        "difficulty": "advanced",
        "source": "cloud",
        "duration_minutes": 100,
        "quality_score": 9.4,
        "completed": True,
    },
    {
        "title": "Production Monitoring Plan",
        "description": "Planned monitoring, logging and cost tracking for a future Weaviate production deployment.",
        "topic": "operations",
        "difficulty": "advanced",
        "source": "planning",
        "duration_minutes": 55,
        "quality_score": 7.8,
        "completed": False,
    },
]

In [6]:
result = events.data.insert_many(course_events)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [7]:
response = events.query.fetch_objects(
    limit=20,
    return_properties=[
        "title",
        "topic",
        "difficulty",
        "source",
        "duration_minutes",
        "quality_score",
        "completed",
    ],
)

for obj in response.objects:
    print(obj.properties)

{'completed': True, 'topic': 'hybrid_search', 'title': 'Hybrid Search', 'quality_score': 9.1, 'source': 'cloud', 'duration_minutes': 65, 'difficulty': 'intermediate'}
{'completed': True, 'topic': 'cross_references', 'title': 'Cross-References', 'quality_score': 8.8, 'source': 'cloud', 'duration_minutes': 80, 'difficulty': 'advanced'}
{'completed': True, 'duration_minutes': 85, 'title': 'Multi-Tenancy', 'quality_score': 9.2, 'source': 'cloud', 'topic': 'multi_tenancy', 'difficulty': 'advanced'}
{'completed': False, 'topic': 'operations', 'title': 'Production Monitoring Plan', 'quality_score': 7.8, 'source': 'planning', 'duration_minutes': 55, 'difficulty': 'advanced'}
{'completed': True, 'topic': 'image_search', 'title': 'Text-to-Image Search', 'quality_score': 9.4, 'source': 'cloud', 'duration_minutes': 90, 'difficulty': 'advanced'}
{'completed': True, 'topic': 'ai_enrichment', 'title': 'AI Enrichment Pipeline', 'quality_score': 9.4, 'source': 'cloud', 'duration_minutes': 100, 'difficu

In [8]:
response = events.aggregate.over_all(total_count=True)

print("Total objects:", response.total_count)

Total objects: 15


In [9]:
response = events.aggregate.over_all(
    total_count=True,
    filters=Filter.by_property("completed").equal(True),
)

print("Completed objects:", response.total_count)

Completed objects: 14


In [10]:
response = events.aggregate.over_all(
    return_metrics=Metrics("duration_minutes").integer(
        count=True,
        minimum=True,
        maximum=True,
        mean=True,
        sum_=True,
    )
)

duration = response.properties["duration_minutes"]

print("Count:", duration.count)
print("Min:", duration.minimum)
print("Max:", duration.maximum)
print("Mean:", duration.mean)
print("Sum:", duration.sum_)

Count: 15
Min: 35
Max: 110
Mean: 70.33333333333333
Sum: 1055


In [11]:
response = events.aggregate.over_all(
    return_metrics=Metrics("quality_score").number(
        count=True,
        minimum=True,
        maximum=True,
        mean=True,
        sum_=True,
    )
)

quality = response.properties["quality_score"]

print("Count:", quality.count)
print("Min:", quality.minimum)
print("Max:", quality.maximum)
print("Mean:", quality.mean)
print("Sum:", quality.sum_)

Count: 15
Min: 7.8
Max: 9.6
Mean: 8.9
Sum: 133.5


In [13]:
response = events.aggregate.over_all(
    return_metrics=Metrics("topic").text(
        top_occurrences_count=True,
        top_occurrences_value=True,
        limit=20,
    )
)

item = response.properties["topic"].top_occurrences[0]

print(type(item))
print(dir(item))
print(item)

<class 'weaviate.collections.classes.aggregate.TopOccurrence'>
['__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__replace__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', 'count', 'value']
TopOccurrence(count=1, value='ai_enrichment')


In [14]:
response = events.aggregate.over_all(
    return_metrics=Metrics("topic").text(
        top_occurrences_count=True,
        top_occurrences_value=True,
        limit=20,
    )
)

for item in response.properties["topic"].top_occurrences:
    print(item.value, "->", item.count)

ai_enrichment -> 1
bm25 -> 1
cloud -> 1
cross_references -> 1
crud -> 1
docker -> 1
generative_search -> 1
hybrid_search -> 1
image_search -> 1
multi_tenancy -> 1
named_vectors -> 1
operations -> 1
rag -> 1
recommendations -> 1
vector_search -> 1


In [15]:
response = events.aggregate.over_all(
    return_metrics=Metrics("difficulty").text(
        top_occurrences_count=True,
        top_occurrences_value=True,
        limit=20,
    )
)

for item in response.properties["difficulty"].top_occurrences:
    print(item.value, "->", item.count)

advanced -> 8
intermediate -> 4
beginner -> 3


In [16]:
response = events.aggregate.over_all(
    group_by=GroupByAggregate(prop="difficulty"),
)

for group in response.groups:
    print("Difficulty:", group.grouped_by.value)
    print("Count:", group.total_count)
    print("-" * 80)

Difficulty: advanced
Count: 8
--------------------------------------------------------------------------------
Difficulty: intermediate
Count: 4
--------------------------------------------------------------------------------
Difficulty: beginner
Count: 3
--------------------------------------------------------------------------------


In [17]:
response = events.aggregate.over_all(
    group_by=GroupByAggregate(prop="source"),
)

for group in response.groups:
    print("Source:", group.grouped_by.value)
    print("Count:", group.total_count)
    print("-" * 80)

Source: cloud
Count: 13
--------------------------------------------------------------------------------
Source: local
Count: 1
--------------------------------------------------------------------------------
Source: planning
Count: 1
--------------------------------------------------------------------------------


In [18]:
response = events.aggregate.over_all(
    total_count=True,
    filters=Filter.by_property("difficulty").equal("advanced"),
    return_metrics=Metrics("duration_minutes").integer(
        count=True,
        minimum=True,
        maximum=True,
        mean=True,
        sum_=True,
    ),
)

duration = response.properties["duration_minutes"]

print("Advanced total:", response.total_count)
print("Advanced duration count:", duration.count)
print("Advanced min:", duration.minimum)
print("Advanced max:", duration.maximum)
print("Advanced mean:", duration.mean)
print("Advanced sum:", duration.sum_)

Advanced total: 8
Advanced duration count: 8
Advanced min: 55
Advanced max: 110
Advanced mean: 86.25
Advanced sum: 690


In [19]:
response = events.aggregate.near_text(
    query="advanced Weaviate features for production applications",
    object_limit=8,
    return_metrics=Metrics("duration_minutes").integer(
        count=True,
        minimum=True,
        maximum=True,
        mean=True,
        sum_=True,
    ),
)

duration = response.properties["duration_minutes"]

print("Semantic aggregate count:", duration.count)
print("Min duration:", duration.minimum)
print("Max duration:", duration.maximum)
print("Mean duration:", duration.mean)
print("Total duration:", duration.sum_)

Semantic aggregate count: 8
Min duration: 35
Max duration: 110
Mean duration: 69.375
Total duration: 555


In [20]:
response = events.aggregate.hybrid(
    query="RAG search vectors cloud",
    alpha=0.5,
    object_limit=8,
    return_metrics=Metrics("quality_score").number(
        count=True,
        minimum=True,
        maximum=True,
        mean=True,
        sum_=True,
    ),
)

quality = response.properties["quality_score"]

print("Hybrid aggregate count:", quality.count)
print("Min quality:", quality.minimum)
print("Max quality:", quality.maximum)
print("Mean quality:", quality.mean)
print("Total quality:", quality.sum_)

Hybrid aggregate count: 9
Min quality: 8.2
Max quality: 9.6
Mean quality: 9.066666666666666
Total quality: 81.6


In [21]:
response = events.query.near_text(
    query="RAG vector search and generative answers",
    limit=5,
    return_properties=[
        "title",
        "topic",
        "difficulty",
        "duration_minutes",
        "quality_score",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Difficulty:", obj.properties["difficulty"])
    print("Duration:", obj.properties["duration_minutes"])
    print("Quality:", obj.properties["quality_score"])
    print("-" * 80)

Distance: 0.5104787945747375
Title: Generative Search
Topic: generative_search
Difficulty: intermediate
Duration: 70
Quality: 9.3
--------------------------------------------------------------------------------
Distance: 0.569189727306366
Title: Focused RAG
Topic: rag
Difficulty: advanced
Duration: 110
Quality: 9.6
--------------------------------------------------------------------------------
Distance: 0.6115347146987915
Title: Vector Search
Topic: vector_search
Difficulty: intermediate
Duration: 60
Quality: 9.0
--------------------------------------------------------------------------------
Distance: 0.6127650141716003
Title: Semantic Recommendations
Topic: recommendations
Difficulty: advanced
Duration: 95
Quality: 9.5
--------------------------------------------------------------------------------
Distance: 0.6327548623085022
Title: Named Vectors
Topic: named_vectors
Difficulty: advanced
Duration: 75
Quality: 9.0
---------------------------------------------------------------------

In [22]:
response = events.aggregate.near_text(
    query="RAG vector search and generative answers",
    object_limit=5,
    return_metrics=Metrics("duration_minutes").integer(
        count=True,
        mean=True,
        sum_=True,
    ),
)

duration = response.properties["duration_minutes"]

print("Objects included:", duration.count)
print("Mean duration:", duration.mean)
print("Total duration:", duration.sum_)

Objects included: 5
Mean duration: 82.0
Total duration: 410


In [23]:
def analyze_topic(query: str, *, object_limit: int = 5) -> None:
    response = events.aggregate.near_text(
        query=query,
        object_limit=object_limit,
        return_metrics=[
            Metrics("duration_minutes").integer(
                count=True,
                minimum=True,
                maximum=True,
                mean=True,
                sum_=True,
            ),
            Metrics("quality_score").number(
                count=True,
                minimum=True,
                maximum=True,
                mean=True,
                sum_=True,
            ),
        ],
    )

    duration = response.properties["duration_minutes"]
    quality = response.properties["quality_score"]

    print("QUERY:", query)
    print("OBJECT LIMIT:", object_limit)
    print("-" * 80)

    print("Duration count:", duration.count)
    print("Duration min:", duration.minimum)
    print("Duration max:", duration.maximum)
    print("Duration mean:", duration.mean)
    print("Duration sum:", duration.sum_)

    print("-" * 80)

    print("Quality count:", quality.count)
    print("Quality min:", quality.minimum)
    print("Quality max:", quality.maximum)
    print("Quality mean:", quality.mean)
    print("Quality sum:", quality.sum_)

In [24]:
analyze_topic(
    "semantic search vector embeddings hybrid search",
    object_limit=6,
)

QUERY: semantic search vector embeddings hybrid search
OBJECT LIMIT: 6
--------------------------------------------------------------------------------
Duration count: 6
Duration min: 40
Duration max: 95
Duration mean: 70.0
Duration sum: 420
--------------------------------------------------------------------------------
Quality count: 6
Quality min: 8.2
Quality max: 9.5
Quality mean: 9.083333333333334
Quality sum: 54.5


In [25]:
analyze_topic(
    "RAG notebooks markdown notes generated answers",
    object_limit=6,
)

QUERY: RAG notebooks markdown notes generated answers
OBJECT LIMIT: 6
--------------------------------------------------------------------------------
Duration count: 6
Duration min: 45
Duration max: 110
Duration mean: 82.5
Duration sum: 495
--------------------------------------------------------------------------------
Quality count: 6
Quality min: 8.0
Quality max: 9.6
Quality mean: 9.133333333333333
Quality sum: 54.8


In [26]:
analyze_topic(
    "SaaS isolation tenants multi tenancy cloud architecture",
    object_limit=6,
)

QUERY: SaaS isolation tenants multi tenancy cloud architecture
OBJECT LIMIT: 6
--------------------------------------------------------------------------------
Duration count: 6
Duration min: 35
Duration max: 85
Duration mean: 63.333333333333336
Duration sum: 380
--------------------------------------------------------------------------------
Quality count: 6
Quality min: 7.8
Quality max: 9.2
Quality mean: 8.666666666666666
Quality sum: 52.0


In [27]:
analytics_response = events.aggregate.near_text(
    query="advanced Weaviate features for real applications",
    object_limit=8,
    return_metrics=[
        Metrics("duration_minutes").integer(
            count=True,
            mean=True,
            sum_=True,
        ),
        Metrics("quality_score").number(
            count=True,
            mean=True,
        ),
    ],
)

duration = analytics_response.properties["duration_minutes"]
quality = analytics_response.properties["quality_score"]

analytics_summary = {
    "object_count": duration.count,
    "mean_duration_minutes": duration.mean,
    "total_duration_minutes": duration.sum_,
    "mean_quality_score": quality.mean,
}

analytics_summary

{'object_count': 8,
 'mean_duration_minutes': 69.375,
 'total_duration_minutes': 555,
 'mean_quality_score': 8.8375}

In [28]:
response = events.generate.near_text(
    query="advanced Weaviate features for real applications",
    grouped_task=(
        "Summarize the retrieved Weaviate learning events. "
        "Explain which advanced concepts appear most useful for real applications. "
        "Mention RAG, named vectors, cross-references, multi-tenancy or recommendations if present."
    ),
    limit=8,
    return_properties=[
        "title",
        "description",
        "topic",
        "difficulty",
        "duration_minutes",
        "quality_score",
    ],
)

print(response.generated)

print("\nAnalytics summary:")
print(analytics_summary)

print("\nSources:")
for obj in response.objects:
    print("-", obj.properties["title"], "|", obj.properties["topic"])

The Weaviate learning events encompass a variety of activities targeting different skill levels, from beginner to advanced. Here’s a summary of the key events and their relevance to real applications:

1. **Docker Weaviate Setup**: This beginner-level event involved starting a Weaviate instance using Docker and connecting it from Python notebooks. It highlights the importance of basic setup for users new to deploying applications.

2. **Production Monitoring Plan**: This event emphasizes planning for monitoring, logging, and cost tracking in a productive Weaviate deployment. Understanding this advanced concept is crucial for managing real-world applications effectively and ensuring performance at scale.

3. **Text-to-Image Search**: In this advanced session, users utilized local CLIP embeddings and self-provided vectors for searching images in Weaviate Cloud. This illustrates the application of machine learning in managing and finding visual data efficiently, making it extremely useful

In [29]:
client.close()